### 1.读取并检查所有数据

In [15]:
import pandas as pd

def clean_monthly_ts(df, value_col):
    """
    df: index = date 的 DataFrame
    value_col: 要保留的数值列名（如 'cn_cpi' / 'cn_pmi'）
    """
    out = df.copy()

    # 强制数值化（把空字符串等变成 NaN）
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")

    # 删除无效日期
    out = out[~out.index.isna()]

    # 关键：如果同一日期出现多行，合并去重（优先保留非空值）
    out = (
        out.sort_index()
           .groupby(level=0)[value_col]
           .apply(lambda s: s.dropna().iloc[0] if not s.dropna().empty else pd.NA)
           .to_frame()
    )

    return out

# ========== 1. 读取 FRED 数据 ==========
fred = pd.read_csv("fred_raw_data.csv", parse_dates=["date"])
fred.set_index("date", inplace=True)
fred.sort_index(inplace=True)

print("FRED columns:", fred.columns.tolist())
print(fred.head())

FRED columns: ['CNY_USD', 'USD_INDEX', 'US_10Y', 'US_CPI']
            CNY_USD  USD_INDEX  US_10Y  US_CPI
date                                          
1947-01-01      NaN        NaN     NaN   21.48
1947-02-01      NaN        NaN     NaN   21.62
1947-03-01      NaN        NaN     NaN   22.00
1947-04-01      NaN        NaN     NaN   22.00
1947-05-01      NaN        NaN     NaN   21.95


### 2.读取中国 CPI（月度）

In [16]:
# ========== 2. 中国 CPI（月度） ==========
cpi_raw = pd.read_csv(
    "china_CPI_month.csv",
    encoding="gbk",
    header=None
)

# 找到“指标”所在行（一般是第 2 行，保险写法）
indicator_row = cpi_raw[cpi_raw.iloc[:, 0] == "指标"].index[0]

# 取指标行下面的数据
cpi_data = cpi_raw.iloc[indicator_row + 1:]

# 重设列名（月份）
cpi_data.columns = cpi_raw.iloc[indicator_row]

# 只保留 CPI 这一行
cpi_row = cpi_data[
    cpi_data["指标"] == "居民消费价格指数(上年同月=100)"
]

# 横板 → 竖板
cpi_long = cpi_row.melt(
    id_vars="指标",
    var_name="date",
    value_name="cn_cpi"
)

# 日期处理
cpi_long["date"] = pd.to_datetime(cpi_long["date"])
cpi_long = cpi_long.set_index("date")[["cn_cpi"]].sort_index()
cpi_long = clean_monthly_ts(cpi_long, "cn_cpi")

print("China CPI OK:", cpi_long.head())



China CPI OK:             cn_cpi
date              
1998-01-01   100.3
1998-02-01    99.9
1998-03-01   100.7
1998-04-01    99.7
1998-05-01    99.0


### 3.读取中国 PMI（月度）

In [17]:
# ========== 3. 中国 PMI（月度） ==========
pmi_raw = pd.read_csv(
    "china_PMI_month.csv",
    encoding="gbk",
    header=None
)

indicator_row = pmi_raw[pmi_raw.iloc[:, 0] == "指标"].index[0]
pmi_data = pmi_raw.iloc[indicator_row + 1:]
pmi_data.columns = pmi_raw.iloc[indicator_row]

pmi_row = pmi_data[
    pmi_data["指标"] == "制造业采购经理指数(%)"
]

pmi_long = pmi_row.melt(
    id_vars="指标",
    var_name="date",
    value_name="cn_pmi"
)

pmi_long["date"] = pd.to_datetime(pmi_long["date"])
pmi_long = pmi_long.set_index("date")[["cn_pmi"]].sort_index()
pmi_long = clean_monthly_ts(pmi_long, "cn_pmi")

print("China PMI OK:", pmi_long.head())



China PMI OK:             cn_pmi
date              
2005-01-01    54.7
2005-02-01    54.5
2005-03-01    57.9
2005-04-01    56.7
2005-05-01    52.9


### 4.月度 → 日度，并合并为主数据表

In [18]:
# ========== 4. 合并月度宏观数据 ==========
monthly = pd.concat([cpi_long, pmi_long], axis=1)

# 月度 → 日度（向前填充）
monthly_daily = monthly.resample("D").ffill()

# ========== 5. 合并到 FRED 日度数据 ==========
master = fred.join(monthly_daily, how="left")
master.index.name = "date"
master = master.sort_index()

# 保存最终主数据
master.to_csv("master_data.csv")

print("✅ master_data.csv 已成功生成")
print(master.head())
print(master.tail())


✅ master_data.csv 已成功生成
            CNY_USD  USD_INDEX  US_10Y  US_CPI  cn_cpi  cn_pmi
date                                                          
1947-01-01      NaN        NaN     NaN   21.48     NaN     NaN
1947-02-01      NaN        NaN     NaN   21.62     NaN     NaN
1947-03-01      NaN        NaN     NaN   22.00     NaN     NaN
1947-04-01      NaN        NaN     NaN   22.00     NaN     NaN
1947-05-01      NaN        NaN     NaN   21.95     NaN     NaN
            CNY_USD  USD_INDEX  US_10Y  US_CPI  cn_cpi  cn_pmi
date                                                          
2025-12-05   7.0696   121.0615    4.14     NaN     NaN     NaN
2025-12-08      NaN        NaN    4.17     NaN     NaN     NaN
2025-12-09      NaN        NaN    4.18     NaN     NaN     NaN
2025-12-10      NaN        NaN    4.13     NaN     NaN     NaN
2025-12-11      NaN        NaN    4.14     NaN     NaN     NaN
